In [ ]:
# Cell 1: AI predictions. Running this cell makes model calls.
from pathlib import Path
import sys

BASE = Path(r"D:\CI_sense_DATA\Final_data")
sys.path.insert(0, str(BASE))
import importlib
import b0_parallel
importlib.reload(b0_parallel)  # Pick up fixes even in an already-running kernel.
from b0_parallel import run_parallel

SELECTED_IDS = None  # None = all 300 B0 rows; IDs never go to the model.
N_RUNS = 3              # Set to 3 for the full repeated experiment.
MODEL = "Gemini 3.8 Flash (Medium)"  # Verify label with `agy models` after sign-in.
EFFORT = "medium"
EXPERIMENT = "b0_flash_medium_v2"  # New name if prompt/model/data changes.
CLI_PATH = None         # None discovers agy.exe automatically.
LOGIN_BEFORE_RUN = True # Offers a real terminal for browser-code sign-in.

# First use selected IDs as a pilot. Check /usage in Antigravity before
# expanding to 900 requests. Quota exhaustion saves and offers account sign-in.
# Fresh conversations, no resumed context; web/tool permissions are denied.
RESULTS_DIR = run_parallel(
    BASE, selected_ids=SELECTED_IDS, n_runs=N_RUNS, model=MODEL,
    effort=EFFORT, experiment=EXPERIMENT, cli=CLI_PATH,
    login_before_run=LOGIN_BEFORE_RUN, workers=6,
)
print("Saved results:", RESULTS_DIR)


In [ ]:
# Cell 2: compare saved predictions with actual labels; no model calls.
from pathlib import Path
import sys
from IPython.display import display

BASE = Path(r"D:\CI_sense_DATA\Final_data")
sys.path.insert(0, str(BASE))
from b0_runner import compare_results

# Uses cell 1's EXPERIMENT when available; can also run independently.
EXPERIMENT = globals().get("EXPERIMENT", "b0_flash_medium_v2")
THRESHOLD = 0.5
comparison, metrics, aggregate, variability = compare_results(BASE, EXPERIMENT, THRESHOLD)
print("Per-test comparison (1 = flaky, 0 = non-flaky):")
display(comparison[["run", "id", "actual", "flaky_probability", "predicted", "correct", "few_shot", "status"]])
print("Per-run metrics; confusion matrix fields are tn, fp, fn, tp:")
display(metrics)
print("Across-run mean/std (std is undefined for a single run):")
display(aggregate)
print("Per-test probability variability:")
display(variability)
